In [1]:

"""このスクリプトは、Gemmaモデルの特定の層におけるEOSトークン位置の隠れ状態を抽出し、PCAで次元削減してプロットするものです。

- 固有名詞リスト(proper_nouns)の各テキストの末尾にEOSトークンを追加して、その位置の隠れ状態を抽出します。(LAST_TOKEN_IS_EOS=Trueの場合)
- 抽出した隠れ状態をPCAで2次元に削減し、プロットします。
- プロットはoutput/hidden_state_pca_plotsディクトリに保存されます。

使用方法:
1. モデルのバージョンとサイズを設定します。
2. 固有名詞リスト(proper_nouns)を必要に応じて変更します。
3. スクリプトを実行して、プロットを生成します。

"""

import os
import json

import torch
import numpy as np
import matplotlib.pyplot as plt
import japanize_matplotlib
import pandas as pd
import plotly.express as px

from transformers import AutoTokenizer, AutoModelForCausalLM
from sklearn.decomposition import PCA



project_root = os.path.join(os.getcwd(), "..")


# =========================
# EOS位置の hidden state を取る関数
# =========================
@torch.no_grad()
def extract_eos_hidden_states(text_list, batch_size=8, layer_index=-1):
    """
    各テキストの末尾にEOSを明示的に追加し、
    EOSトークン位置の hidden state を返す。

    Returns:
        np.ndarray of shape (N, hidden_dim)
    """
    all_vecs = []

    for i in range(0, len(text_list), batch_size):
        batch_texts = text_list[i:i + batch_size]

        # EOS を明示的に末尾へ追加
        batch_texts_with_eos = [text + tokenizer.eos_token for text in batch_texts]

        inputs = tokenizer(
            batch_texts_with_eos,
            return_tensors="pt",
            padding=True,
            truncation=True,
            add_special_tokens=LAST_TOKEN_IS_EOS,
        ).to(model.device) 

        input_ids = inputs["input_ids"]
        attention_mask = inputs["attention_mask"]

        with torch.no_grad():
            outputs = model(**inputs, output_hidden_states=True)
            # hidden_states は tuple:
            # 0: embedding出力, 1..L: 各層出力
            hs = outputs.hidden_states
            layer_hs = hs[layer_index]      # (B, T, H)
        
        
        # 各系列について EOS token の最後の出現位置を取る
        eos_mask = (input_ids == tokenizer.eos_token_id)

        for b in range(input_ids.size(0)):
            if LAST_TOKEN_IS_EOS:
                eos_positions = torch.where(eos_mask[b])[0]
                if len(eos_positions) == 0:
                    raise ValueError(f"EOS token が見つかりません: {batch_texts[b]}")
                eos_pos = eos_positions[-1].item()

                vec = layer_hs[b, eos_pos, :]   # (H,)
            else:
                seq_len = attention_mask[b].sum().item()
                vec = layer_hs[b, seq_len - 1, :]  # (H,)

            all_vecs.append(vec.detach().float().cpu().numpy())

    return np.stack(all_vecs, axis=0)


/home/toko/project/EmbedNewConcept-20260305/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:

# =========================
# 設定
# =========================


model_version = "3"
model_size = "4"
LAST_TOKEN_IS_EOS = True  # 固有名詞の最後にEOSトークンを追加して、その位置のhidden stateを取るかどうか。Falseの場合は最終トークン位置の隠れ状態を取得する
CUDA_VISIBLE_DEVICES = "2"
output_dir = os.path.join(project_root, "src_visualize", "output", "hidden_state_pca_plots")


MODEL_NAME = model_name = f"google/gemma-{model_version}-{model_size}b-it"

os.environ["CUDA_VISIBLE_DEVICES"] = CUDA_VISIBLE_DEVICES
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

# 可視化したい固有名詞リスト
category_to_proper_nouns = {
    "都市": ["東京", "大阪", "京都", "渋谷"],
    "企業": ["ソニー", "トヨタ", "任天堂", "OpenAI"],
    "地理": ["富士山", "北海道"],
}

# どの層の hidden state を使うか
# -1 なら最終層
LAYER_INDEX = 9

# バッチサイズ
BATCH_SIZE = 8


# =========================
# フラット化
# =========================
proper_nouns = []
categories = []

for category, nouns in category_to_proper_nouns.items():
    for noun in nouns:
        proper_nouns.append(noun)
        categories.append(category)



# =========================
# モデル読み込み
# =========================
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float32,
    # torch_dtype=torch.float16 if DEVICE == "cuda" else torch.float32,
    device_map=None
).to(DEVICE)
model.eval()

# Gemma系で pad token が未設定な場合の保険
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

if tokenizer.eos_token_id is None:
    raise ValueError("tokenizer.eos_token_id が見つかりません。")



# =========================
# 特徴抽出
# =========================
features = extract_eos_hidden_states(
    proper_nouns,
    batch_size=BATCH_SIZE,
    layer_index=LAYER_INDEX
)


print("features shape:", features.shape)  # (N, hidden_dim)

# NaN / inf 除外
valid_mask = np.isfinite(features).all(axis=1)
valid_features = features[valid_mask]
valid_proper_nouns = [x for x, ok in zip(proper_nouns, valid_mask) if ok]
valid_categories = [x for x, ok in zip(categories, valid_mask) if ok]

print("valid samples:", len(valid_proper_nouns), "/", len(proper_nouns))

if len(valid_proper_nouns) < 2:
    raise ValueError("有効サンプルが少なすぎて PCA できません。")



# =========================
# PCA
# =========================
pca = PCA(n_components=3)
coords = pca.fit_transform(valid_features)

print("Explained variance ratio:", pca.explained_variance_ratio_)


# =========================
# DataFrame化
# =========================
df = pd.DataFrame({
    "PC1": coords[:, 0],
    "PC2": coords[:, 1],
    "PC3": coords[:, 2],
    "label": valid_proper_nouns,
    "category": valid_categories,
})


# =========================
# Plot
# =========================
fig = px.scatter_3d(
    df,
    x="PC1",
    y="PC2",
    z="PC3",
    color="category",
    hover_name="label",
    hover_data={"category": True, "PC1": ':.3f', "PC2": ':.3f', "PC3": ':.3f'},
    title=f"3D PCA of EOS hidden states ({MODEL_NAME}, layer={LAYER_INDEX}, LAST_TOKEN_IS_EOS={LAST_TOKEN_IS_EOS})",
)
fig.update_traces(marker=dict(size=6))
fig.show()

`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|██████████| 883/883 [00:01<00:00, 866.20it/s] 


features shape: (10, 2560)
valid samples: 10 / 10
Explained variance ratio: [0.7959028  0.07431468 0.03002441]
